In [ ]:
import os,sys,json

In [ ]:
#request_id = 2918
request_id = 2929
url_link = f"https://pandamon01.sdcc.bnl.gov/tasks/?idds_request_id={request_id}&json"

curl_command = f"curl -H 'Accept: application/json' -H 'Content-Type: application/json' \"{url_link}\""
print(curl_command)

In [ ]:
import subprocess
outfile_name = f"idds_request_id_{request_id}.txt"
output=""
if not os.path.exists(outfile_name):
    result = subprocess.run(curl_command,shell=True,capture_output=True,text=True)
    output=result.stdout
else:
    with open(outfile_name) as of:
        output = of.read()


In [ ]:
output = result.stdout
print("STDOUT: ",output)
print("STDERR: ",result.stderr)

In [ ]:
import requests

headers = {
    "Accept": "application/json",
    "Content-Type": "application/json"
}

response = requests.get(url_link,headers=headers)

In [ ]:
data = response.json()
print(type(data))

In [ ]:
print(json.dumps(data,indent=4))

In [ ]:
if isinstance(data,dict):
    for key in data.keys():
        print("Top level key ",key )
elif isinstance(data,list):
    print(f"Top level object with list of length {len(data)} elements")
    if len(data)>0 and isinstance(data[0],dict):
        print("keys inside the first element: ")
        for key in data[0].keys():
            print(" ",key)

In [ ]:
for item in data:
    print("taskname: ",item.get("taskname"))
    print(f"cpu time {item.get('cputime')} in units {item.get('cputimeunit')}")
    print(f"wall time {item.get('walltime')} in units {item.get('walltimeunit')}")
    print(f"start time {item.get('starttime')} & end time {item.get('endtime')}")

In [ ]:
from collections import defaultdict
import re
from datetime import datetime,timedelta
trial_times = defaultdict(lambda: {"start": [], "end": [], "duration": []})
FMT="%Y-%m-%dT%H:%M:%S.%f"
for item in data:
    taskname = item.get("taskname")
    start = item.get("starttime")
    end = item.get("endtime")
    if not taskname:
        continue
    match = re.search(r"trial_(\d+)",taskname)
    if not match:
        continue
    trial_num = match.group(1)
    if start:
        trial_times[trial_num]['start'].append(start)
        trial_times[trial_num]['end'].append(end)
        duration = datetime.strptime(end,FMT) - datetime.strptime(start,FMT)
        trial_times[trial_num]['duration'].append(duration.total_seconds())
trial_times = dict(trial_times)

In [ ]:
print(json.dumps(trial_times,indent=4))

In [ ]:
# now lets look at what the json file has to say
#file = "../scheduler_epic/drich_multistep_multi_obj_1000evts_10trial_threshold.json"
#file = "../scheduler_epic/drich_multistep_multi_obj_5000evts_30trial_threshold.json"
file = "../scheduler_epic/drich_multistep_multi_obj_5000evts_50trial_threshold_2obj.json"
f = open(file,'r')
jdata = json.load(f)
exp = jdata['experiment']
trials = exp['trials']
t0 = trials['0']
t0.keys()


In [ ]:
c1,c2,r1 = t0['time_created'],t0['time_completed'],t0['time_run_started']
print(c1,c2,r1)

In [ ]:
JFMT = "%Y-%m-%d %H:%M:%S.%f"
c = datetime.strptime(c1['value'],JFMT)

In [ ]:
c = datetime.strptime(c2['value'],JFMT) - datetime.strptime(c1['value'],JFMT)


In [ ]:
c.total_seconds()

In [ ]:
#What i want to do is get the difference between the min of time start and max of time end and get the difference...
trial_times = defaultdict(lambda: {"start": [], "end": [],"modification": [], "duration": []})
FMT="%Y-%m-%dT%H:%M:%S.%f"
for item in data:
    taskname = item.get("taskname")
    start = item.get("starttime")
    modification = item.get("modificationtime")
    end = item.get("endtime")
    if not taskname:
        continue
    match = re.search(r"trial_(\d+)",taskname)
    if not match:
        continue
    trial_num = match.group(1)
    #print(trial_num,taskname)
    if start:
        trial_times[trial_num]['start'].append(datetime.strptime(start,FMT))
        trial_times[trial_num]['end'].append(datetime.strptime(end,FMT))
        trial_times[trial_num]['modification'].append(datetime.strptime(modification,FMT))
        duration = datetime.strptime(end,FMT) - datetime.strptime(start,FMT)
        trial_times[trial_num]['duration'].append(duration.total_seconds())
trial_times = dict(trial_times)

In [ ]:
trial_times.keys()

In [ ]:
max(trial_times['0']['start']),trial_times['0']['start']

In [ ]:
duration_panda= {}
start_panda = {}
task_duration = {}
for t in trial_times:
    trial = trial_times[t]
    st_list = trial['start']
    end_list = trial['end']
    task_duration[t] = max(trial['duration']) 
    start = min(st_list)
    start_panda[t] = start
    end = max(end_list)
    tot_duration = (end-start)
    duration_panda[t] = tot_duration

  

In [ ]:
task_duration

In [ ]:
for t in trial_times:
    trial = trial_times[t]
    st_list = trial['start']
    end_list = trial['end']
    mod_list = trial['modification']
    if t=='5':
        print(min(st_list),max(end_list))

In [ ]:
#now same for jsone data....
trials.keys()

In [ ]:
JFMT = "%Y-%m-%d %H:%M:%S.%f"

def coerce_dt(v):
    return datetime.strptime(v,JFMT) if isinstance(v,str) else v

def coerce_td(v):
    if isinstance(v,timedelta):
        return v
    if isinstance(v,(int,float)):
        return timedelta(seconds=float(v))
    raise TypeError(f"Unsupported type: {type(v)}")
    


duration_ax={}
start_ax = {}
for t in trials:
    tr = trials[t]
    st = coerce_dt(tr['time_created']['value'])
    end = coerce_dt(tr['time_completed']['value'])
    dur = end-st
    start_ax[t] = st
    duration_ax[t] = dur
    if t=='5':
        print(tr['time_created']['value'],tr['time_completed']['value'])

#normalize the values from panda

# Normalize PanDA dicts to datetime/timedelta
for t in start_panda:
    start_panda[t] = coerce_dt(start_panda[t])
for t in duration_panda:
    #duration_panda basically gives the life time of a task in the PanDA that includes time in between jobs within tasks as well
    #duration_panda[t] = coerce_td(duration_panda[t])
    duration_panda[t] = coerce_td(task_duration[t])
    

In [ ]:
# Use a consistent y index per trial (sorted by trial key for stability)
trial_ids = sorted(set(start_ax.keys()) & set(start_panda.keys()))
trial_to_idx = {t: i for i, t in enumerate(trial_ids)}

t0 = min(start_ax[t] for t in trial_ids)

def hours_since_t0(dt):
    return(dt-t0).total_seconds()/3600.0

def td_to_hours(td):
    return(td.total_seconds())/3600.0



In [ ]:
import plotly.graph_objects as go

In [ ]:

fig = go.Figure()

# --- AX traces (blue) ---
for i in range(0,len(trial_ids)):
    t = str(i)
    y = int(i)
    x0_h = hours_since_t0(start_ax[t])
    x1_h = x0_h + td_to_hours(duration_ax[t])
    fig.add_trace(go.Scatter(
        x=[x0_h, x1_h],
        y=[y, y],
        mode="lines",
        line=dict(width=6,color="red"),  # default blue
        name="AX (with overheads)",
        hovertemplate=(
            f"Trial: {t}<br>"
            "Type: AX (with overheads)<br>"
            "Start (h): %{x:.3f}<br>"
            f"Index: {y}<br>"
            f"Duration (h): {td_to_hours(duration_ax[t]):.3f}<extra></extra>"
        ),
        showlegend=False  # avoid legend per segment; add one dummy entry below
    ))

In [ ]:
# --- PanDA traces (black) ---
#for t in trial_ids:
for i in range(0,len(trial_ids)):
    t = str(i)
    y = int(i)
    x0_h = hours_since_t0(start_ax[t])
    x1_h = x0_h + td_to_hours(duration_panda[t])
    fig.add_trace(go.Scatter(
        x=[x0_h, x1_h],
        y=[y, y],
        mode="lines",
        line=dict(width=6, color="black"),
        name="PanDA (core runtime)",
        hovertemplate=(
            f"Trial: {t}<br>"
            "Type: PanDA (core runtime)<br>"
            "Start: %{x:.3f}<br>"
            f"Index: {y}<br>"
            f"Duration (h): {td_to_hours(duration_panda[t]):.3f}<extra></extra>"
        ),
        showlegend=False
    ))

# Single legend entries (dummy traces) for clarity
fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", line=dict(width=6,color="red"), name="AX (with overheads)"))
fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", line=dict(width=6, color="black"), name="PanDA (core runtime)"))

fig.update_layout(
    title="Trial Duration vs Elapsed Time",
    xaxis=dict(title="Elapsed Time (hour)"),
    yaxis=dict(title="Trial Index"),
    hovermode="closest",
    showlegend=True,
    height=700
)
fig.update_xaxes(range=[0,25])
fig.show()